In [1]:
import wbgapi as wb
import pandas as pd

In [2]:
df_exp = wb.data.DataFrame("NE.EXP.GNFS.CD", time=range(2010, 2026), labels=True, skipAggs=True)
df_imp= wb.data.DataFrame("NE.IMP.GNFS.CD", time=range(2010, 2026), labels=True, skipAggs=True)
PIB_pays = wb.data.DataFrame("NY.GDP.MKTP.CD", time=range(2010, 2026), labels=True)


In [3]:
PIB_world = PIB_pays[PIB_pays["Country"] == "World"]

In [4]:
df_exp.rename(
    columns={f"YR{i}": f"YR{i}_exp" for i in range(2010, 2026)}, inplace=True
)
df_imp.rename(
    columns={f"YR{i}": f"YR{i}_imp" for i in range(2010, 2026)}, inplace=True
)


In [5]:
df_full = pd.merge(
    df_exp,
    df_imp,
    on=["economy", "Country"],
    how="outer",
)

In [6]:
for i in range(2010, 2026):
    df_full[f"YR{i}_balance"] = (
        df_full[f"YR{i}_exp"] - df_full[f"YR{i}_imp"]
    )
df_balance = df_full.drop(
    columns=[f"YR{i}_exp" for i in range(2010, 2026)]
    + [f"YR{i}_imp" for i in range(2010, 2026)]
)

In [7]:
df_balance_norm = df_balance.copy()
for i in range(2010, 2026):
    df_balance_norm[f"YR{i}_balance"] = (
        df_balance[f"YR{i}_balance"] * 100 / PIB_world.loc["WLD"][f"YR{i}"]
    )

In [8]:
df_balance_norm["YR2025_balance"].sum()

np.float64(1.5305680308420873)

In [9]:
df_balance_norm.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/df_balance_norm.csv", index=False
)